In [1]:
import pandas as pd
from collections import defaultdict

# 读取 CSV
csv_path = r"F:\NWP\S1_S2_matched\matched_satellite_images_1.csv"
df = pd.read_csv(csv_path)
print("原始数据概览:")
print(f"总行数: {len(df)}")
print(f"状态分布: {df['status'].value_counts().to_dict()}")
print(f"传感器分布: {df['matched_sensor'].value_counts().to_dict()}")
print()

# 只处理成功匹配的数据
df_success = df[df['status'] == 'success'].copy()
print(f"成功匹配的数据行数: {len(df_success)}")

# 处理空值和数据类型
df_success['matched_sensor'] = df_success['matched_sensor'].fillna("").astype(str)


原始数据概览:
总行数: 899
状态分布: {'success': 848, 'no_match': 51}
传感器分布: {'Sentinel-2': 742, 'Landsat-8': 106}

成功匹配的数据行数: 848


In [2]:
# 按S1图像分组，收集对应的S2和L8图像
pairs = defaultdict(lambda: {"s2": [], "l8": []})

for _, row in df_success.iterrows():
    s1_id = row['s1_image_id']  # 已经包含完整路径
    matched_id = row['matched_image_id']
    sensor = row['matched_sensor'].upper()
    
    if sensor.startswith("SENTINEL-2"):
        pairs[s1_id]["s2"].append(matched_id)
    elif sensor.startswith("LANDSAT-8"):
        pairs[s1_id]["l8"].append(matched_id)

print(f"\n找到 {len(pairs)} 个独特的S1图像")


找到 156 个独特的S1图像


In [3]:
# 生成JavaScript代码用于GEE
js_code = []
js_code.append("// SAR-Optical Image Pairs for Google Earth Engine")
js_code.append("// Generated from matched satellite images data")
js_code.append("// Usage: Use these IDs to filter and visualize matched images in GEE\n")

pair_count = 0
for i, (s1_id, matched_ids) in enumerate(pairs.items(), 1):
    if matched_ids["s2"] or matched_ids["l8"]:  # 确保有匹配的图像
        pair_count += 1
        js_code.append(f"// ==================== Pair {pair_count} ====================")
        js_code.append(f"// S1 Image ID")
        js_code.append(f"var s1_id_{pair_count} = '{s1_id}';")
        js_code.append("")
        
        # Sentinel-2 IDs
        js_code.append(f"// Corresponding Sentinel-2 Images ({len(matched_ids['s2'])} images)")
        js_code.append(f"var s2_ids_{pair_count} = [")
        for j, s2_id in enumerate(matched_ids["s2"]):
            comma = "," if j < len(matched_ids["s2"]) - 1 else ""
            js_code.append(f"  '{s2_id}'{comma}")
        js_code.append("];")
        js_code.append("")
        
        # Landsat-8 IDs
        js_code.append(f"// Corresponding Landsat-8 Images ({len(matched_ids['l8'])} images)")
        js_code.append(f"var l8_ids_{pair_count} = [")
        for j, l8_id in enumerate(matched_ids["l8"]):
            comma = "," if j < len(matched_ids["l8"]) - 1 else ""
            js_code.append(f"  '{l8_id}'{comma}")
        js_code.append("];")
        js_code.append("")

# 添加使用示例
js_code.append("// ==================== Usage Example ====================")
js_code.append("// Example: Load first pair of images")
js_code.append("/*")
js_code.append("var s1_collection = ee.ImageCollection('COPERNICUS/S1_GRD')")
js_code.append("  .filter(ee.Filter.eq('system:id', s1_id_1));")
js_code.append("")
js_code.append("var s2_collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')")
js_code.append("  .filter(ee.Filter.inList('system:id', s2_ids_1));")
js_code.append("")
js_code.append("var l8_collection = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')")
js_code.append("  .filter(ee.Filter.inList('system:id', l8_ids_1));")
js_code.append("")
js_code.append("// Visualize")
js_code.append("Map.addLayer(s1_collection.first().select('VV'), {min: -20, max: 0}, 'SAR VV');")
js_code.append("Map.addLayer(s2_collection.first().select(['B4', 'B3', 'B2']), {min: 0, max: 3000}, 'Sentinel-2 RGB');")
js_code.append("*/")

# 保存到文件
output_content = "\n".join(js_code)

# 保存为.js文件（用于GEE）
js_filename = r"F:\NWP\S1_S2_matched\gee_sar_optical_pairs.js"
with open(js_filename, "w", encoding="utf-8") as f:
    f.write(output_content)

# 保存为.txt文件（方便阅读和编辑）
txt_filename = r"F:\NWP\S1_S2_matched\gee_sar_optical_pairs.txt"
with open(txt_filename, "w", encoding="utf-8") as f:
    f.write(output_content)

# 额外生成一个简化的配对表格文件
summary_lines = []
summary_lines.append("SAR-光学图像配对汇总表")
summary_lines.append("=" * 60)
summary_lines.append(f"生成时间: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append("")

for i, (s1_id, matched_ids) in enumerate(pairs.items(), 1):
    if matched_ids["s2"] or matched_ids["l8"]:
        summary_lines.append(f"配对 {i}:")
        summary_lines.append(f"  SAR: {s1_id}")
        summary_lines.append(f"  S2 数量: {len(matched_ids['s2'])}")
        summary_lines.append(f"  L8 数量: {len(matched_ids['l8'])}")
        summary_lines.append("")

summary_filename = r"F:\NWP\S1_S2_matched\sar_optical_pairs_summary.txt"
with open(summary_filename, "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines))

print(f"生成了 {pair_count} 对SAR-光学图像匹配")
print("\n统计信息:")
s2_total = sum(len(ids["s2"]) for ids in pairs.values())
l8_total = sum(len(ids["l8"]) for ids in pairs.values())
print(f"- 总共 {s2_total} 个Sentinel-2匹配")
print(f"- 总共 {l8_total} 个Landsat-8匹配")

print(f"\n📁 文件已保存:")
print(f"  1. {js_filename} - GEE JavaScript代码（可直接在GEE中使用）")
print(f"  2. {txt_filename} - 文本格式（方便阅读和编辑）")
print(f"  3. {summary_filename} - 配对汇总表")

# 显示生成的代码（前500行用于预览）
print(f"\n生成的JavaScript代码预览（总共{len(js_code)}行）:")
print("=" * 50)
print("\n".join(js_code[:30]))  # 显示前30行
if len(js_code) > 30:
    print("... (省略中间部分) ...")
    print("\n".join(js_code[-10:]))  # 显示最后10行

print("\n✅ 处理完成！")
print("📁 代码已生成，可以复制到GEE中使用")
print("💡 每个S1图像都有对应的s2_ids和l8_ids数组，方便在GEE中筛选和可视化")

生成了 156 对SAR-光学图像匹配

统计信息:
- 总共 742 个Sentinel-2匹配
- 总共 106 个Landsat-8匹配

📁 文件已保存:
  1. F:\NWP\S1_S2_matched\gee_sar_optical_pairs.js - GEE JavaScript代码（可直接在GEE中使用）
  2. F:\NWP\S1_S2_matched\gee_sar_optical_pairs.txt - 文本格式（方便阅读和编辑）
  3. F:\NWP\S1_S2_matched\sar_optical_pairs_summary.txt - 配对汇总表

生成的JavaScript代码预览（总共2739行）:
// SAR-Optical Image Pairs for Google Earth Engine
// Generated from matched satellite images data
// Usage: Use these IDs to filter and visualize matched images in GEE

// ==================== Pair 1 ====================
// S1 Image ID
var s1_id_1 = 'COPERNICUS/S1_GRD/S1A_EW_GRDM_1SDH_20230329T155110_20230329T155214_047861_05C03E_8378';

// Corresponding Sentinel-2 Images (6 images)
var s2_ids_1 = [
  'COPERNICUS/S2_SR_HARMONIZED/20230329T210051_20230329T210046_T10XDM',
  'COPERNICUS/S2_SR_HARMONIZED/20230329T210051_20230329T210046_T10XEM',
  'COPERNICUS/S2_SR_HARMONIZED/20230329T210051_20230329T210046_T10XEN',
  'COPERNICUS/S2_SR_HARMONIZED/20230329T210051_20230329